In [2]:
"""
MLB GUMBO API real-time websocket and REST live feed client.
Optimized for execution within an active Jupyter Notebook / ipykernel environment.
"""

import asyncio
import logging
import json
from typing import Any, Dict
import aiohttp
import websockets

# Configure structured logging for Notebook streaming output
# Re-initialize handlers to prevent duplicate lines in notebook cell stdout
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger("GumboNotebookClient")

# Test parameters
TEST_GAME_PK = 717325  
PING_INTERVAL_SECONDS = 60
REST_BASE_URL = "https://statsapi.mlb.com/api/v1.1/game/{game_pk}/feed/live"
WS_BASE_URL = "wss://ws.statsapi.mlb.com/api/v1/game/push/subscribe/gameday/{game_pk}"


async def fetch_gumbo_state(session: aiohttp.ClientSession, game_pk: int, timestamp: str) -> None:
    """
    Queries the REST feed layer using the precise event timestamp parsed
    from the WebSocket push frame to pull down the un-truncated JSON layout.
    """
    url = REST_BASE_URL.format(game_pk=game_pk)
    params = {"timestamp": timestamp}
    
    try:
        async with session.get(url, params=params, timeout=10) as response:
            if response.status == 200:
                data: Dict[str, Any] = await response.json()
                game_data = data.get("gameData", {})
                live_data = data.get("liveData", {})
                linescore = live_data.get("linescore", {})
                
                logger.info(
                    f"Successfully fetched GUMBO payload | "
                    f"Status: {game_data.get('status', {}).get('abstractGameState')} | "
                    f"Inning: {linescore.get('currentInningOrdinal', 'N/A')} | "
                    f"Runs: H {linescore.get('teams', {}).get('home', {}).get('runs', 0)} - "
                    f"A {linescore.get('teams', {}).get('away', {}).get('runs', 0)}"
                )
            else:
                logger.error(f"Failed to fetch GUMBO state payload. HTTP Status: {response.status}")
    except Exception as e:
        logger.error(f"Error occurring during REST fetch task: {str(e)}")


async def send_heartbeat(websocket: websockets.ClientConnection) -> None:
    """
    Looping background task keeping the channel active via the standard 'Gameday5' push frame.
    Updated to use modern websockets.ClientConnection type hinting.
    """
    while True:
        try:
            await asyncio.sleep(PING_INTERVAL_SECONDS)
            logger.debug("Dispatching 'Gameday5' heartbeat sequence...")
            await websocket.send("Gameday5")
        except asyncio.CancelledError:
            break
        except Exception as e:
            logger.error(f"Heartbeat transmitter experienced a connection fault: {str(e)}")
            break


async def stream_gumbo_events(game_pk: int) -> None:
    """
    Main state machine orchestrating socket handshake, ping loop, and event delegation.
    """
    ws_url = WS_BASE_URL.format(game_pk=game_pk)
    logger.info(f"Connecting to WebSocket endpoint: {ws_url}")
    
    async with aiohttp.ClientSession() as http_session:
        try:
            async with websockets.connect(ws_url) as websocket:
                logger.info(f"WebSocket channel established for gamePk: {game_pk}")
                
                # Spin up background heartbeat worker task
                heartbeat_task = asyncio.create_task(send_heartbeat(websocket))
                
                try:
                    async for message in websocket:
                        if message == "Gameday5" or not isinstance(message, str):
                            continue
                        
                        try:
                            payload = json.loads(message)
                            timestamp = payload.get("timeStamp")
                            events = payload.get("gameEvents", [])
                            logical_events = payload.get("logicalEvents", [])
                            
                            logger.info(
                                f"Push Message Received | UpdateID: {payload.get('updateId')} | "
                                f"Events: {events} | Logical: {logical_events}"
                            )
                            
                            if timestamp:
                                asyncio.create_task(fetch_gumbo_state(http_session, game_pk, timestamp))
                                
                        except json.JSONDecodeError:
                            logger.warning(f"Raw frame data could not be parsed to JSON schema: {message}")
                            
                finally:
                    heartbeat_task.cancel()
                    await asyncio.gather(heartbeat_task, return_exceptions=True)
                    
        except Exception as e:
            logger.critical(f"WebSocket connection encountered a fatal error or disconnected: {str(e)}")


# Execution context tailored explicitly for ipykernel's running event loop.
logger.info(f"Initializing testing protocol sequence for game context: {TEST_GAME_PK}")
try:
    # We await the coroutine directly because Jupyter runs inside an active loop.
    await stream_gumbo_events(TEST_GAME_PK)
except asyncio.CancelledError:
    logger.info("Asynchronous task sequence canceled.")

2026-06-15 10:02:04 [INFO] Initializing testing protocol sequence for game context: 717325
2026-06-15 10:02:04 [INFO] Connecting to WebSocket endpoint: wss://ws.statsapi.mlb.com/api/v1/game/push/subscribe/gameday/717325
2026-06-15 10:02:04 [INFO] WebSocket channel established for gamePk: 717325
2026-06-15 10:02:04 [CRITICAL] WebSocket connection encountered a fatal error or disconnected: received 4400 (private use) Game is not available for subscription at this time.; then sent 4400 (private use) Game is not available for subscription at this time.


In [5]:
"""
MLB GUMBO API Historical Data Extraction Script.
Optimized for direct execution within a Jupyter Notebook cell.
"""

import logging
from typing import Any, Dict
import requests

# Re-initialize logging handlers to prevent duplicate rows inside standard notebook output
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(
    level=logging.INFO, 
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)
logger = logging.getLogger("GumboHistoricalClient")

HISTORICAL_GAME_PK = 717325  
HISTORICAL_URL = f"https://statsapi.mlb.com/api/v1.1/game/{HISTORICAL_GAME_PK}/feed/live"


def extract_historical_gumbo(game_pk: int) -> None:
    """
    Queries the final historical GUMBO REST state frame and safely
    unpacks game meta parameters along with deep pitch matrices.
    """
    logger.info(f"Requesting historical GUMBO payload for gamePk: {game_pk}")
    
    try:
        response = requests.get(HISTORICAL_URL, timeout=15)
        
        # Fixed: requests utilizes .status_code instead of aiohttp's .status
        if response.status_code == 200:
            gumbo_data: Dict[str, Any] = response.json()
            
            # 1. Structural Layer: gameData (Teams, Venue, Status, Weather)
            game_data = gumbo_data.get("gameData", {})
            teams = game_data.get("teams", {})
            home_team = teams.get("home", {}).get("name", "Unknown Home")
            away_team = teams.get("away", {}).get("name", "Unknown Away")
            
            # 2. Structural Layer: liveData (Boxscore, Linescore, Intersected Plays)
            live_data = gumbo_data.get("liveData", {})
            linescore = live_data.get("linescore", {})
            
            # Extract cumulative scores safely
            home_runs = linescore.get("teams", {}).get("home", {}).get("runs", 0)
            away_runs = linescore.get("teams", {}).get("away", {}).get("runs", 0)
            
            logger.info("--- GAME METADATA RECOVERY ---")
            logger.info(f"Matchup: {away_team} vs {home_team}")
            logger.info(f"Final Score: {away_team} {away_runs} - {home_team} {home_runs}")
            logger.info(f"Total Innings Tracked: {linescore.get('currentInningOrdinal', 'N/A')}")
            
            # 3. Deep Historical Parse: Extracting structural tracking matrices
            all_plays = live_data.get("plays", {}).get("allPlays", [])
            logger.info(f"Successfully processed {len(all_plays)} complete plate appearances (at-bats).")
            
            if all_plays:
                logger.info("--- SAMPLE AT-BAT DATA EXTRACTION ---")
                sample_play = all_plays[0]  # The initial at-bat sequence of the match
                result = sample_play.get("result", {})
                about = sample_play.get("about", {})
                count = sample_play.get("count", {})
                
                logger.info(f"Inning: {about.get('halfInning', 'N/A').upper()} {about.get('inning', 1)}")
                logger.info(f"Event: {result.get('event')} | Description: {result.get('description')}")
                logger.info(f"Final At-Bat Count: {count.get('balls')}B / {count.get('strikes')}S")
                
                # Sub-array layer containing absolute tracking vectors for every pitch delivered
                pitch_events = sample_play.get("playEvents", [])
                logger.info(f"Total measurement frames (pitches/pickoffs) in this plate appearance: {len(pitch_events)}")
                
        else:
            logger.error(f"Failed to query historical endpoint. HTTP Status Code: {response.status_code}")
            
    except requests.exceptions.RequestException as e:
        logger.error(f"Network transport level error occurred during collection sequence: {str(e)}")


# Execute extraction sequence
extract_historical_gumbo(HISTORICAL_GAME_PK)

2026-06-15 10:03:21 [INFO] Requesting historical GUMBO payload for gamePk: 717325
2026-06-15 10:03:21 [INFO] --- GAME METADATA RECOVERY ---
2026-06-15 10:03:21 [INFO] Matchup: St. Louis Cardinals vs Chicago Cubs
2026-06-15 10:03:21 [INFO] Final Score: St. Louis Cardinals 3 - Chicago Cubs 4
2026-06-15 10:03:21 [INFO] Total Innings Tracked: 9th
2026-06-15 10:03:21 [INFO] Successfully processed 75 complete plate appearances (at-bats).
2026-06-15 10:03:21 [INFO] --- SAMPLE AT-BAT DATA EXTRACTION ---
2026-06-15 10:03:21 [INFO] Inning: TOP 1
2026-06-15 10:03:21 [INFO] Event: Strikeout | Description: Dylan Carlson strikes out swinging.
2026-06-15 10:03:21 [INFO] Final At-Bat Count: 1B / 3S
2026-06-15 10:03:21 [INFO] Total measurement frames (pitches/pickoffs) in this plate appearance: 7


In [10]:
# Place this code directly into a cell in a Jupyter Notebook.
# Ensure you have 'requests' installed in your active environment: !pip install requests

import requests
from typing import Dict, Any, Optional

def test_season_gumbo_presence(year: int) -> None:
    """
    Queries the schedule for the given year, extracts the first completed game,
    and inspects the GUMBO live feed payload to report field population state.
    """
    session = requests.Session()
    session.headers.update({
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36",
        "Accept": "application/json"
    })
    
    # 1. Fetch the schedule to find a valid completed game ID (gamePk)
    schedule_url = "https://statsapi.mlb.com/api/v1/schedule"
    params = {
        "sportId": 1,
        "startDate": f"{year}-04-01",
        "endDate": f"{year}-10-01",
    }
    
    print(f"Checking Season {year}...")
    try:
        schedule_res = session.get(schedule_url, params=params, timeout=10)
        if schedule_res.status_code != 200:
            print(f"  -> Error: Unable to fetch schedule for {year} (Status: {schedule_res.status_code})")
            return
            
        schedule_data = schedule_res.json()
        target_game_pk: Optional[int] = None
        
        # Locate the first final game
        for date_node in schedule_data.get("dates", []):
            for game in date_node.get("games", []):
                if game.get("status", {}).get("abstractGameState") == "Final":
                    target_game_pk = game.get("gamePk")
                    break
            if target_game_pk:
                break
                
        if not target_game_pk:
            print(f"  -> Error: No completed games located in the schedule for {year}.")
            return
            
        print(f"  -> Selected Sample Game ID: {target_game_pk}")
        
        # 2. Query the GUMBO live feed endpoint for this game
        gumbo_url = f"https://statsapi.mlb.com/api/v1.1/game/{target_game_pk}/feed/live"
        gumbo_res = session.get(gumbo_url, timeout=15)
        
        if gumbo_res.status_code != 200:
            print(f"  -> Error: GUMBO endpoint returned status {gumbo_res.status_code}")
            return
            
        gumbo_data = gumbo_res.json()
        live_data = gumbo_data.get("liveData", {})
        plays_node = live_data.get("plays", {})
        all_plays = plays_node.get("allPlays", [])
        
        # 3. Analyze payload structures
        total_plays = len(all_plays)
        print(f"  -> Total records found in 'allPlays': {total_plays}")
        
        if total_plays > 0:
            # Check if individual play records contain nested tracking entries
            sample_play = all_plays[0]
            play_events = sample_play.get("playEvents", [])
            print(f"  -> Sample play event array size: {len(play_events)}")
            print(f"  -> Status: VERIFIED POPULATED")
        else:
            print(f"  -> Status: EMPTY PAYLOAD (Skeletal tracking structure)")
            
    except Exception as e:
        print(f"  -> Exception occurred during extraction: {str(e)}")
    print("-" * 50)

# Run the troubleshooting loop across the transition threshold years
for test_year in range(1930, 2011):
    test_season_gumbo_presence(test_year)

Checking Season 1930...
  -> Selected Sample Game ID: 105564
  -> Total records found in 'allPlays': 0
  -> Status: EMPTY PAYLOAD (Skeletal tracking structure)
--------------------------------------------------
Checking Season 1931...
  -> Selected Sample Game ID: 106805
  -> Total records found in 'allPlays': 0
  -> Status: EMPTY PAYLOAD (Skeletal tracking structure)
--------------------------------------------------
Checking Season 1932...
  -> Selected Sample Game ID: 108034
  -> Total records found in 'allPlays': 0
  -> Status: EMPTY PAYLOAD (Skeletal tracking structure)
--------------------------------------------------
Checking Season 1933...
  -> Selected Sample Game ID: 109268
  -> Total records found in 'allPlays': 0
  -> Status: EMPTY PAYLOAD (Skeletal tracking structure)
--------------------------------------------------
Checking Season 1934...
  -> Selected Sample Game ID: 110494
  -> Total records found in 'allPlays': 0
  -> Status: EMPTY PAYLOAD (Skeletal tracking structu

KeyboardInterrupt: 

In [14]:
!python --version

Python 3.11.1
